# JSONB 데이터 타입 처리 및 인덱싱 최적화 — 실습 노트북

> 5교시(직접 재봅니다) Step 0~4, 6교시(필수 실습), 7교시(🔰 미션·보너스)를 여기서 실제로 실행합니다.
> 오늘 앞 파트 **"PostgreSQL 기초 및 고급 쿼리 작성 실습"** 과 **"윈도우 함수 심화"** 세션에서 띄운
> `db-pg` 컨테이너와 `course_db`를 그대로 씁니다 — 별도 설치나 컨테이너 재기동이 필요 없습니다.

## 실행 안내
- 🐳 **선행 조건**: `db-pg` 컨테이너가 이미 켜져 있어야 합니다.
- 🔐 비밀번호는 코드에 하드코딩하지 않고 같은 폴더 `.env` 파일(`PGPASSWORD=postgres`)에서 `load_dotenv()`로 읽습니다.
- 위에서 아래로 **순서대로** 실행하세요. `documents`는 **`course_db`에 실제로 만드는 영구 테이블**이라
  다음 주 pgvector 시간에 `embedding VECTOR(1024)` 컬럼이 여기에 그대로 추가됩니다.
  `doc_meta`는 인덱스 실험 전용 연습 테이블이라 이후 세션에서 이어지지 않습니다.

In [1]:
# ✅ 포인트: psycopg = Python용 PostgreSQL 드라이버. 오늘 앞 파트(PostgreSQL 기초·윈도우함수)와 동일한 버전을 씁니다.
# 🆕 psycopg 'v3'를 씁니다(import psycopg). 과거의 psycopg2와 다릅니다.
#
# 📌 %pip install: 주피터 노트북 안에서 파이썬 패키지를 설치하는 명령입니다.
#   -q(quiet)는 설치 로그를 최소화하고, "psycopg[binary]==3.3.4"는 정확히 이 버전을 설치하라는 뜻입니다
#   (==으로 버전을 고정하면 다른 버전 때문에 코드가 안 되는 문제를 예방할 수 있습니다).
#   python-dotenv는 .env 파일에서 비밀번호 같은 값을 읽어오는 패키지입니다(바로 다음 셀에서 사용).
%pip install -q "psycopg[binary]==3.3.4" python-dotenv
import psycopg
print("psycopg 버전:", psycopg.__version__)

Note: you may need to restart the kernel to use updated packages.
psycopg 버전: 3.3.4


## 접속 — psycopg & .env

오늘 만든 `documents`·`doc_meta`는 `course_db` 안에 만듭니다 — `user_events`·`log_events`와 같은
데이터베이스에 나란히 쌓입니다. 비밀번호는 `.env`의 `PGPASSWORD`에서 읽습니다(하드코딩 금지).

In [2]:
import os
from dotenv import load_dotenv

# load_dotenv(): 같은 폴더의 .env 파일을 읽어 그 안의 KEY=VALUE 값들을 파이썬 환경변수로 등록합니다.
# 비밀번호를 코드에 직접 적지 않기 위한 표준적인 방법입니다.
load_dotenv()                      # 같은 폴더의 .env → 환경변수
pw = os.environ["PGPASSWORD"]      # 비밀번호는 .env 파일에 (하드코딩 금지)

# psycopg.connect(...): PostgreSQL 서버에 실제로 접속합니다.
#   host/port: 접속 위치(내 컴퓨터의 도커 컨테이너, 기본 포트 5432)
#   dbname="course_db": 접속할 데이터베이스 이름(오늘 1교시에 만든 것을 그대로 씁니다)
#   user/password: 로그인 계정과 비밀번호
conn = psycopg.connect(host="localhost", port=5432, dbname="course_db", user="postgres", password=pw)
# cur: 커서(cursor) — SQL을 실제로 실행하고 결과를 받아오는 창구. 이 노트북에서는 계속 재사용합니다.
cur = conn.cursor()
print("연결 완료 →", conn)

연결 완료 → <psycopg.Connection [IDLE] (host=localhost user=postgres database=course_db) at 0x2d4937a9e80>


In [3]:
# 실습 편의용 출력 헬퍼 — 교안의 "예상 결과" 표와 같은 모양으로 찍어줍니다.
# (SQL 문법과는 무관한 순수 파이썬 코드로, 결과를 표 모양으로 예쁘게 보여주기 위한 도구일 뿐입니다.)

def show(rows, cols):
    # None(=SQL의 NULL)은 화면에 빈 문자열로 바꿔서 보여줍니다.
    rows = [tuple("" if v is None else v for v in r) for r in rows]
    # 각 컬럼에서 "가장 긴 값의 글자수"를 구해 표의 칸 너비로 사용합니다(줄이 깔끔하게 맞도록).
    widths = [max(len(str(c)), *(len(str(r[i])) for r in rows)) if rows else len(str(c))
              for i, c in enumerate(cols)]
    print(" | ".join(str(c).ljust(w) for c, w in zip(cols, widths)))  # 헤더(컬럼명) 줄
    print("-+-".join("-" * w for w in widths))                        # 구분선
    for r in rows:
        print(" | ".join(str(r[i]).ljust(w) for i, w in enumerate(widths)))  # 데이터 행들

def run(sql, params=None):
    # SELECT 실행 + 결과 반환 (컬럼명 포함)
    cur.execute(sql, params)                       # SQL을 실제로 데이터베이스에 보내 실행
    cols = [d.name for d in cur.description]        # 결과 컬럼 이름들만 추출
    return cur.fetchall(), cols                     # (행 목록, 컬럼명 목록)을 함께 반환

def explain(sql):
    # EXPLAIN 실행 계획을 줄 단위로 출력
    # EXPLAIN(ANALYZE 없이)은 "PostgreSQL이 이 쿼리를 어떻게 실행할 계획인지"를 예측치로만
    # 보여줍니다(실제로 실행하지는 않음) — Seq Scan(순차 스캔)인지 Index Scan(인덱스 스캔)인지가
    # 이 절의 핵심 관찰 포인트입니다.
    cur.execute("EXPLAIN " + sql)
    for (line,) in cur.fetchall():   # 결과가 한 줄짜리 튜플들의 목록이라 (line,) 형태로 풀어서 받습니다.
        print(line)

---
## 2교시 · 왜 정규화 스키마가 버티지 못하는가

정규화 스키마는 모든 데이터를 미리 정해진 열에 담습니다. 데이터가 균일할 때는 강력하지만, 문서마다 필드가 다른 순간 무너집니다. 작은 데모로 이 붕괴 과정을 직접 겪어봅니다. (교안 2교시 모듈 2-1)

In [4]:
# DROP TABLE IF EXISTS: 이미 있으면 먼저 지웁니다(반복 실행 안전장치).
cur.execute("DROP TABLE IF EXISTS doc_normalized_demo")
# "정규화" 방식: 모든 필드를 미리 정해진 컬럼(열)으로 선언합니다.
cur.execute('''
    CREATE TEMP TABLE doc_normalized_demo (
        id SERIAL PRIMARY KEY,   -- 자동 증가 고유번호
        author TEXT,
        company TEXT,
        doc_type TEXT
    )
''')
cur.execute('''
    INSERT INTO doc_normalized_demo (author, company, doc_type)
    VALUES ('김영수', '삼성', '사업보고서')
''')

# 두 번째 문서 유형(감사보고서)이 등장하니 revision·approver라는 새 필드가 필요해졌습니다.
# ALTER TABLE ... ADD COLUMN: 이미 존재하는 테이블에 컬럼을 하나 추가하는 DDL(스키마 변경) 명령입니다.
# 이 명령을 실행하면 "기존에 이미 있던 행들"에는 이 새 컬럼 값이 전부 NULL로 채워집니다.
cur.execute("ALTER TABLE doc_normalized_demo ADD COLUMN revision TEXT")
cur.execute("ALTER TABLE doc_normalized_demo ADD COLUMN approver TEXT")
cur.execute('''
    INSERT INTO doc_normalized_demo (author, company, doc_type, revision, approver)
    VALUES ('이미영', 'LG', '감사보고서', '1차', '박부장')
''')

# 특수 유형(financial_year·auditor)이 또 등장 → 또 다시 ALTER TABLE이 필요합니다.
# 이렇게 새 문서 유형이 나타날 때마다 스키마를 계속 바꿔야 하는 것이 "정규화 스키마의 한계"입니다.
cur.execute("ALTER TABLE doc_normalized_demo ADD COLUMN financial_year INT")
cur.execute("ALTER TABLE doc_normalized_demo ADD COLUMN auditor TEXT")
cur.execute('''
    INSERT INTO doc_normalized_demo (author, company, doc_type, financial_year, auditor)
    VALUES ('박지수', '현대', '특수보고서', 2025, '삼정KPMG')
''')
conn.commit()

rows, cols = run("SELECT * FROM doc_normalized_demo ORDER BY id")
show(rows, cols)
# 문서 유형이 늘어날 때마다 ALTER TABLE이 필요했고, 대부분의 행에서 대부분의 열이 NULL입니다.

id | author | company | doc_type | revision | approver | financial_year | auditor
---+--------+---------+----------+----------+----------+----------------+--------
1  | 김영수    | 삼성      | 사업보고서    |          |          |                |        
2  | 이미영    | LG      | 감사보고서    | 1차       | 박부장      |                |        
3  | 박지수    | 현대      | 특수보고서    |          |          | 2025           | 삼정KPMG 


문서 유형이 하나 늘어날 때마다 `ALTER TABLE`이 필요했고, 결과 테이블은 대부분의 행에서 대부분의 열이 `NULL`인 희소 테이블이 됐습니다. 이것이 **컬럼 폭증**입니다.

- 정규화 테이블: DB가 타입을 강제하고 통계 정보로 쿼리 최적화가 정확하지만, 필드가 늘어날 때마다 DDL이 필요합니다.
- JSONB: 타입 강제가 없고 옵티마이저 추정이 부정확할 수 있지만, 새 키를 그냥 넣으면 됩니다.

JSONB는 이 예측 불가능성을 테이블 밖으로 밀어내는 전략입니다 — 뒤이어 3교시에서 어떻게 저장되는지 봅니다. (모듈 2-2)

---
## 3교시 · JSON과 JSONB, 그리고 연산자

PostgreSQL에는 JSON을 다루는 타입이 두 가지입니다. `json`은 입력을 원문 그대로 보존하고, `jsonb`는 파싱된 이진 형태로 저장합니다. 같은 데이터를 두 타입에 넣고 차이를 직접 확인합니다. (교안 3교시 모듈 3-1)

In [ ]:
cur.execute("DROP TABLE IF EXISTS test_json")
# 한 테이블에 JSON 타입 컬럼과 JSONB 타입 컬럼을 나란히 두어 같은 데이터를 넣고 비교합니다.
cur.execute("CREATE TEMP TABLE test_json (data_json JSON, data_jsonb JSONB)")
# 일부러 "순서가 뒤섞이고(b가 a보다 먼저)" "같은 키(a)가 중복된" 값을 넣습니다 —
# json과 jsonb가 이를 각각 어떻게 저장하는지 차이를 보여주기 위한 의도적인 설계입니다.
cur.execute('''
    INSERT INTO test_json VALUES (
        '{ "b": 2, "a": 1, "a": 99 }',
        '{ "b": 2, "a": 1, "a": 99 }'
    )
''')
conn.commit()
rows, cols = run("SELECT data_json, data_jsonb FROM test_json")
show(rows, cols)
# data_json: 입력한 원문 그대로 (키 순서 b,a,a 유지 + 중복 키 a도 둘 다 유지)   #지금 data_json 출력은 a = 1 이 없어져 있긴 함 - 강사님이 수정할 예정
# data_jsonb: 키가 알파벳순으로 정렬되고, 중복 키는 마지막 값(a=99)만 남습니다

data_json         | data_jsonb       
------------------+------------------
{'b': 2, 'a': 99} | {'a': 99, 'b': 2}


| 항목 | `json` | `jsonb` |
|---|---|---|
| 저장 형태 | 텍스트 원문 | 파싱된 이진 |
| 키 순서 | 입력 순서 보존 | 알파벳 정렬 |
| 중복 키 | 모두 유지 | 마지막 값만 |
| 인덱스 | GIN 불가 | GIN 가능 |

실습에서는 항상 `jsonb`를 씁니다 — 이어지는 3교시 모듈 3-2의 연산자 5종(`->`/`->>`/`#>`/`#>>`/`@>`/`?`/`?|`/`?&`)은 바로 다음 Step 0~2에서 `documents` 테이블로 직접 실습합니다.

---
## Step 0 · documents 테이블 — 4건 (중첩 구조 포함)

교안 3교시(연산자 5종)·6교시(필수 실습)·5교시 Step 0에서 공통으로 쓰는 표본 데이터입니다.

In [6]:
# CASCADE: 이 테이블에 의존하는 다른 객체(예: 이 테이블을 참조하는 뷰)가 있으면 함께 지웁니다.
cur.execute("DROP TABLE IF EXISTS documents CASCADE")
# JSONB 타입: 이 컬럼 하나에 구조가 서로 다른 JSON 문서를 자유롭게 담을 수 있습니다
# (정규화 테이블처럼 미리 컬럼을 다 정의해둘 필요가 없습니다 — 2교시에서 본 문제의 해법입니다).
cur.execute("""
    CREATE TABLE documents (
        id SERIAL PRIMARY KEY,
        title TEXT,
        metadata JSONB
    )
""")
# 4건의 문서를 넣습니다. 각 문서의 metadata는 서로 다른 키 구성을 가지고 있습니다
# (예: 3번은 tags·pages가 있고, 4번은 아예 다른 구조로 중첩(info→author→name/dept)되어 있습니다).
# '{"key":"value", ...}' 형태의 작은따옴표로 감싼 문자열이 그대로 JSONB로 저장됩니다.
cur.execute("""
    INSERT INTO documents (title, metadata) VALUES
        ('삼성 사업보고서', '{"author":"김영수","company":"삼성","doc_type":"사업보고서","tags":["공시","연간"]}'),
        ('LG 감사보고서',   '{"author":"이미영","company":"LG","doc_type":"감사보고서","financial_year":2025}'),
        ('현대 반기보고서',  '{"author":"박지수","company":"현대","doc_type":"반기보고서","tags":["공시","반기"],"pages":220}'),
        ('복잡한 문서', '{"info":{"author":{"name":"홍길동","dept":"재무"}}}')
""")
conn.commit()
rows, cols = run("SELECT id, title, metadata FROM documents ORDER BY id")
show(rows, cols)

id | title    | metadata                                                                                   
---+----------+--------------------------------------------------------------------------------------------
1  | 삼성 사업보고서 | {'tags': ['공시', '연간'], 'author': '김영수', 'company': '삼성', 'doc_type': '사업보고서'}              
2  | LG 감사보고서 | {'author': '이미영', 'company': 'LG', 'doc_type': '감사보고서', 'financial_year': 2025}            
3  | 현대 반기보고서 | {'tags': ['공시', '반기'], 'pages': 220, 'author': '박지수', 'company': '현대', 'doc_type': '반기보고서'}
4  | 복잡한 문서   | {'info': {'author': {'dept': '재무', 'name': '홍길동'}}}                                        


---
## Step 1 · `->` / `->>` / `#>` / `#>>` — 값·경로 접근 (교안 3교시 모듈 3-2)

In [7]:
# ① -> : JSON / JSONB 로 반환 (큰따옴표 포함)
# metadata -> 'company' : metadata 안에서 'company' 키의 값을 꺼냅니다. 결과 타입은 여전히 JSONB이므로
# 문자열이라도 화면에는 큰따옴표가 포함된 형태로 나옵니다(예: "삼성"). 다른 JSONB 연산과 계속 연결할 때 씁니다.
rows, cols = run("SELECT title, metadata -> 'company' AS company FROM documents WHERE id <= 3")
show(rows, cols)

title    | company
---------+--------
삼성 사업보고서 | 삼성     
LG 감사보고서 | LG     
현대 반기보고서 | 현대     


In [8]:
# 배열 인덱스 접근
# metadata -> 'tags' 로 배열(JSONB 배열)을 꺼낸 뒤, -> 0 으로 그 배열의 0번째 요소를 꺼냅니다.
# JSONB 배열의 인덱스는 0부터 시작합니다(파이썬 리스트와 같습니다 — SQL의 다른 배열 타입은 1부터 시작하니 헷갈리지 않도록 주의).
rows, cols = run("SELECT metadata -> 'tags' -> 0 AS first_tag FROM documents WHERE id = 1")
show(rows, cols)

first_tag
---------
공시       


In [ ]:
#위 실행 결과를 보지만 말고 다양하게 바꿔보자

In [9]:
# ② ->> : TEXT로 반환 (큰따옴표 없음) — WHERE 비교에 사용
# ->  와의 차이: ->>는 결과를 순수 TEXT(문자열)로 돌려줍니다. 그래서 WHERE 절에서 '감사보고서' 같은
# 일반 문자열과 등호(=)로 직접 비교할 수 있습니다. ->로 꺼낸 JSONB 값은 이런 비교가 안 됩니다.
rows, cols = run("SELECT title, metadata ->> 'company' AS company FROM documents WHERE id <= 3")
show(rows, cols)

rows, cols = run("SELECT title FROM documents WHERE metadata ->> 'doc_type' = '감사보고서'")
show(rows, cols)

title    | company
---------+--------
삼성 사업보고서 | 삼성     
LG 감사보고서 | LG     
현대 반기보고서 | 현대     
title   
--------
LG 감사보고서


In [10]:
# ③ #> / #>> : 중첩 경로 접근 (4번 문서)
# 4번 문서는 metadata.info.author.name 처럼 여러 단계로 중첩되어 있습니다.
# -> 를 여러 번 이어 쓰는 대신, #> 뒤에 경로를 배열 형태 '{키1,키2,키3}'로 한 번에 지정할 수 있습니다.
# #> 는 JSONB로, #>> 는 TEXT로 반환합니다(-> / ->>의 관계와 동일).
rows, cols = run("SELECT metadata #> '{info,author,name}' AS name_jsonb FROM documents WHERE id = 4")
show(rows, cols)

rows, cols = run("SELECT metadata #>> '{info,author,dept}' AS dept_text FROM documents WHERE id = 4")
show(rows, cols)

name_jsonb
----------
홍길동       
dept_text
---------
재무       


---
## Step 2 · `@>` / `?` / `?|` / `?&` — 포함·키 존재 (교안 3교시 모듈 3-2)

In [14]:
# ④ @> : 포함 여부 — GIN 인덱스를 타는 연산자 ★
# metadata @> '{"company":"삼성"}' : "metadata가 이 작은 JSON 조각을 부분집합으로 포함하고 있는가?"를 묻습니다.
# 즉 metadata 안에 company 키가 있고 그 값이 정확히 "삼성"이면 참(true)입니다.
# 이 연산자는 4교시에서 배우는 GIN 인덱스가 최적화해줄 수 있는 핵심 연산자입니다.
rows, cols = run('''SELECT title FROM documents WHERE metadata @> '{"company":"삼성"}' ''')
show(rows, cols)

# 배열 안의 값 하나를 포함하는지도 같은 방식으로 검사할 수 있습니다 — tags 배열에 "반기"가 들어있는 문서 찾기.
rows, cols = run('''SELECT title FROM documents WHERE metadata @> '{"tags":["반기"]}' ''')
show(rows, cols)

title   
--------
삼성 사업보고서
title   
--------
현대 반기보고서


In [15]:
# ⑤ ? / ?| / ?& : 키 존재 여부
# ? 'key' : metadata의 최상위 레벨에 이 키가 존재하는지만 검사합니다(값은 보지 않음).
rows, cols = run("SELECT title FROM documents WHERE metadata ? 'financial_year'")
show(rows, cols)

# ?| ARRAY[...] : 배열 안의 키들 중 "하나라도(OR)" 존재하면 참입니다.
rows, cols = run("SELECT title FROM documents WHERE metadata ?| ARRAY['financial_year', 'pages']")
show(rows, cols)

# ?& ARRAY[...] : 배열 안의 키들이 "모두(AND)" 존재해야 참입니다.
rows, cols = run("SELECT title FROM documents WHERE metadata ?& ARRAY['author', 'company']")
show(rows, cols)

title   
--------
LG 감사보고서
title   
--------
LG 감사보고서
현대 반기보고서
title   
--------
삼성 사업보고서
LG 감사보고서
현대 반기보고서


> ⚠️ **흔한 실수**: `->>` 결과는 항상 TEXT입니다. `WHERE metadata ->> 'pages' > 100`처럼 숫자를
> 문자열로 비교하면 사전순 비교가 되어 틀린 결과가 나옵니다 — `(metadata ->> 'pages')::INT > 100`처럼
> 명시적으로 형변환해야 합니다.

---
## 4교시 · GIN 인덱스는 어떻게 생겼는가

B-tree는 열의 단일 스칼라 값 기준으로 정렬된 구조라 범위·등치 검색에 강하지만, JSONB 내부의 여러 키·값을 동시에 인덱싱할 수는 없습니다. 방금 만든 `documents`에 B-tree를 걸어 직접 확인합니다. (교안 4교시 모듈 4-1)

In [13]:
# CREATE INDEX ... ON documents (metadata) : 일반적인 B-tree 인덱스를 metadata 컬럼 전체에 만듭니다.
# B-tree는 "컬럼 하나의 값을 통째로" 정렬해서 찾는 구조라, 등호(=)나 범위(<, >) 비교에는 강하지만
# JSONB 내부의 개별 키·값까지 파고들어 검색하지는 못합니다 — 아래에서 이를 직접 확인합니다.
cur.execute("CREATE INDEX idx_doc_btree ON documents (metadata)")
conn.commit()

print("=== B-tree 인덱스를 만들어도 @> 조회는 여전히 Seq Scan ===")
explain('''SELECT * FROM documents WHERE metadata @> '{"company":"삼성"}' ''')
# B-tree는 JSONB 열의 "전체 값" 하나만 비교할 수 있어 내부 키 검색에는 쓰이지 못합니다.

cur.execute("DROP INDEX idx_doc_btree")   # 이 데모 전용 인덱스이므로 확인 후 바로 정리합니다.
conn.commit()

=== B-tree 인덱스를 만들어도 @> 조회는 여전히 Seq Scan ===
Seq Scan on documents  (cost=0.00..1.05 rows=1 width=68)
  Filter: (metadata @> '{"company": "삼성"}'::jsonb)


### GIN 역색인 구조 (모듈 4-2)

GIN(Generalized Inverted Index)은 역색인입니다 — "값 → 그 값이 들어있는 행의 목록"을 저장합니다. 책 뒤의 찾아보기와 같은 구조입니다.

```
정방향 (테이블):          역방향 (GIN 인덱스):
행 1 → {a, b, c}          "a" → [1, 3]
행 2 → {b, d}             "b" → [1, 2]
행 3 → {a, c}             "c" → [1, 3]
```

`documents`에 GIN을 걸고, 4건뿐인 작은 테이블에서도 인덱스가 실제로 쓰이는 모습을 확인합니다.

In [16]:
# USING GIN (metadata) : B-tree 대신 GIN(Generalized Inverted Index, 역색인) 방식으로 인덱스를 만듭니다.
# GIN은 JSONB 내부의 각 키·값 조각까지 색인하므로 @> 같은 "포함 여부" 검색에 인덱스를 쓸 수 있습니다.
cur.execute("CREATE INDEX idx_doc_gin ON documents USING GIN (metadata)")
conn.commit()

# SET enable_seqscan = OFF : PostgreSQL 설정을 일시적으로 바꿔 "순차 스캔(Seq Scan)을 쓰지 말라"고
# 강제합니다. 4건뿐인 아주 작은 테이블에서는 옵티마이저가 "인덱스를 타나 안 타나 차이가 없다"고
# 판단해 그냥 Seq Scan을 고를 수 있기 때문에, 인덱스가 실제로 동작하는 모습을 보려고 잠시 꺼두는 것입니다.
cur.execute("SET enable_seqscan = OFF")
print("=== GIN 인덱스 생성 후 (Seq Scan 강제 비활성화) ===")
explain('''SELECT * FROM documents WHERE metadata @> '{"company":"삼성"}' ''')
cur.execute("SET enable_seqscan = ON")   # 원래대로 복구 — 이후 셀에 영향이 없도록 반드시 되돌립니다.
# Bitmap Index Scan on idx_doc_gin이 보이면 GIN이 실제로 값을 찾는 데 쓰인 것입니다.

=== GIN 인덱스 생성 후 (Seq Scan 강제 비활성화) ===
Bitmap Heap Scan on documents  (cost=12.93..16.94 rows=1 width=68)
  Recheck Cond: (metadata @> '{"company": "삼성"}'::jsonb)
  ->  Bitmap Index Scan on idx_doc_gin  (cost=0.00..12.93 rows=1 width=0)
        Index Cond: (metadata @> '{"company": "삼성"}'::jsonb)


<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=localhost user=postgres database=course_db) at 0x2d49348abd0>

### `jsonb_path_ops` — 더 가벼운 GIN (모듈 4-3)

- 기본 GIN(`jsonb_ops`): `@>`, `?`, `?|`, `?&` 모두 지원 — 인덱스 크기가 큼
- `jsonb_path_ops`: `@>` 연산자만 지원 — 인덱스 크기가 작고 `@>` 검색이 더 빠름, 키 존재 검색(`?`)은 불가

`documents`에 두 인덱스를 모두 만들어 크기를 비교합니다.

In [17]:
# USING GIN (metadata jsonb_path_ops) : GIN 인덱스를 만들 때 "연산자 클래스(operator class)"를
# jsonb_path_ops로 지정합니다. 기본값(jsonb_ops)보다 지원하는 연산자는 적지만(@>만 가능, ?/?|/?&는 불가),
# 색인 구조가 더 단순해서 크기가 작고 @> 검색이 더 빠릅니다.
cur.execute("CREATE INDEX idx_doc_path ON documents USING GIN (metadata jsonb_path_ops)")
conn.commit()

# pg_indexes / pg_relation_size / pg_size_pretty : PostgreSQL이 제공하는 시스템 카탈로그·함수들로,
# 실제 데이터가 아니라 "인덱스가 디스크에서 차지하는 크기" 같은 메타정보를 조회할 때 씁니다.
# pg_size_pretty()는 바이트 숫자를 "112 kB" 처럼 사람이 읽기 쉬운 단위로 바꿔줍니다.
rows, cols = run('''
    SELECT indexname, pg_size_pretty(pg_relation_size(indexname::regclass)) AS size
    FROM pg_indexes
    WHERE tablename = 'documents' AND indexname IN ('idx_doc_gin', 'idx_doc_path')
''')
show(rows, cols)
# 4건뿐이라 크기 차이는 미미합니다 — 같은 비교를 5,000행짜리 doc_meta로 다시 하면(Step 4)
# 차이가 훨씬 뚜렷해집니다.

# 이 절 전용으로 만든 데모 인덱스는 정리합니다 (documents는 곧 🔰 미션에서 company 컬럼을 추가로 씁니다)
cur.execute("DROP INDEX IF EXISTS idx_doc_gin")
cur.execute("DROP INDEX IF EXISTS idx_doc_path")
conn.commit()

indexname    | size 
-------------+------
idx_doc_gin  | 16 kB
idx_doc_path | 16 kB


---
## Step 3 · doc_meta 5,000행 + GIN 인덱스 (교안 5교시 모듈 5-1)

4행짜리 `documents`에서는 인덱스 유무 차이가 안 보이므로, 5,000행짜리 `doc_meta`로 넘어갑니다.
Python 반복문 5,000회 대신 SQL `INSERT ... SELECT ... jsonb_build_object(...)`로 한 번에 생성합니다.

In [18]:
# CASCADE: 이 테이블에 의존하는 다른 객체가 있으면 함께 삭제합니다.
cur.execute("DROP TABLE IF EXISTS doc_meta CASCADE")
cur.execute("CREATE TABLE doc_meta (id SERIAL PRIMARY KEY, title TEXT, metadata JSONB)")
# INSERT INTO ... SELECT ... FROM generate_series(1, 5000) :
#   generate_series(1, 5000)는 1부터 5000까지의 정수를 만들어내는 함수입니다 — 이것을 "5000번 반복"의
#   재료로 씁니다. 파이썬으로 for문 5000번 돌리며 INSERT하는 대신, SQL 한 번으로 5000행을 통째로 만듭니다.
# jsonb_build_object('key1', 값1, 'key2', 값2, ...) : 키-값 쌍들을 넘겨 그 자리에서 JSONB 객체를 만드는 함수입니다.
#   (ARRAY['김영수','이미영',...])[1 + floor(random()*5)::int]
#     → 배열에서 무작위로 하나를 뽑는 관용구입니다. random()은 0~1 사이의 실수, *5는 0~5 사이로 확장,
#       floor()는 소수점을 버려 정수로 만들고, ::int로 정수 타입임을 명시합니다. 배열 인덱스가 1부터
#       시작하므로 1을 더해 1~5 범위의 인덱스를 만듭니다.
#   (50 + (random()*300)::int) → 50~350 사이의 무작위 페이지 수를 만듭니다.
cur.execute("""
    INSERT INTO doc_meta (title, metadata)
    SELECT
        '문서_' || s,   -- || 는 PostgreSQL의 문자열 연결(concatenation) 연산자입니다 (파이썬의 + 와 비슷)
        jsonb_build_object(
            'author', (ARRAY['김영수','이미영','박지수','최민호','정하늘'])[1 + floor(random()*5)::int],
            'company', (ARRAY['삼성','LG','현대','SK','롯데','포스코','KT','신한'])[1 + floor(random()*8)::int],
            'doc_type', (ARRAY['사업보고서','감사보고서','반기보고서','분기보고서'])[1 + floor(random()*4)::int],
            'pages', (50 + (random()*300)::int)
        )
    FROM generate_series(1, 5000) s
""")
conn.commit()
rows, cols = run("SELECT count(*) FROM doc_meta")
show(rows, cols)

count
-----
5000 


In [19]:
# 아직 doc_meta에는 어떤 인덱스도 없습니다 — @> 를 쓰더라도 PostgreSQL은 5000행을 처음부터 끝까지
# 다 훑어보는 Seq Scan(순차 스캔)을 할 수밖에 없습니다. 바로 다음 셀에서 GIN 인덱스를 만든 뒤 비교합니다.
print("=== 인덱스 없이: @> 도 Seq Scan ===")
explain('''SELECT * FROM doc_meta WHERE metadata @> '{"company":"삼성"}' ''')

=== 인덱스 없이: @> 도 Seq Scan ===
Seq Scan on doc_meta  (cost=0.00..155.50 rows=808 width=116)
  Filter: (metadata @> '{"company": "삼성"}'::jsonb)


In [20]:
print("=== GIN(기본, jsonb_ops) 생성 후 ===")
cur.execute("CREATE INDEX idx_meta_default ON doc_meta USING GIN (metadata)")
conn.commit()

print("--- @> 는 인덱스를 탄다 ---")
# 실행 계획에 Bitmap Index Scan on idx_meta_default가 보이면, PostgreSQL이 5000행을 다 훑지 않고
# 인덱스를 이용해 필요한 행만 골라냈다는 뜻입니다 — 앞의 Seq Scan보다 훨씬 효율적입니다.
explain('''SELECT * FROM doc_meta WHERE metadata @> '{"company":"삼성"}' ''')

print()
print("--- 그런데 ->> 비교는 여전히 Seq Scan (GIN이 못 도와줌) ---")
# GIN 인덱스는 "@>, ?, ?|, ?& 연산자"를 위한 것이지, ->>로 꺼낸 텍스트 값을 비교하는 것까지
# 도와주지는 못합니다. 그래서 인덱스가 있어도 이 쿼리는 여전히 Seq Scan입니다.
explain("SELECT * FROM doc_meta WHERE metadata ->> 'company' = '삼성'")

=== GIN(기본, jsonb_ops) 생성 후 ===
--- @> 는 인덱스를 탄다 ---
Bitmap Heap Scan on doc_meta  (cost=17.21..120.31 rows=808 width=116)
  Recheck Cond: (metadata @> '{"company": "삼성"}'::jsonb)
  ->  Bitmap Index Scan on idx_meta_default  (cost=0.00..17.01 rows=808 width=0)
        Index Cond: (metadata @> '{"company": "삼성"}'::jsonb)

--- 그런데 ->> 비교는 여전히 Seq Scan (GIN이 못 도와줌) ---
Seq Scan on doc_meta  (cost=0.00..168.00 rows=25 width=116)
  Filter: ((metadata ->> 'company'::text) = '삼성'::text)


**가설 1 확인**: `@>`는 GIN 인덱스를 탑니다(`Bitmap Heap Scan` + `Bitmap Index Scan on idx_meta_default`).
`->>`는 인덱스가 있어도 `Seq Scan`을 합니다.

---
### Step 3 마지막 · 표현식 인덱스로 `->>` 해결하기 (교안 5교시 모듈 5-2)

In [21]:
# CREATE INDEX ... ON doc_meta ((metadata ->> 'company')) :
#   "표현식 인덱스(expression index)"라고 부릅니다. 컬럼 자체가 아니라 "metadata ->> 'company'라는
#   계산식의 결과값"을 인덱싱합니다. 그래서 나중에 똑같은 표현식으로 조회하면(WHERE metadata ->> 'company' = ...)
#   이 인덱스를 탈 수 있게 됩니다 — 앞 셀에서 GIN이 해결하지 못했던 ->> 비교 문제의 해법입니다.
cur.execute("CREATE INDEX idx_meta_company ON doc_meta ((metadata ->> 'company'))")
conn.commit()

explain("SELECT * FROM doc_meta WHERE metadata ->> 'company' = '삼성'")
# → Bitmap Heap Scan + Bitmap Index Scan on idx_meta_company (더 이상 Seq Scan 아님)

Bitmap Heap Scan on doc_meta  (cost=4.48..62.54 rows=25 width=116)
  Recheck Cond: ((metadata ->> 'company'::text) = '삼성'::text)
  ->  Bitmap Index Scan on idx_meta_company  (cost=0.00..4.47 rows=25 width=0)
        Index Cond: ((metadata ->> 'company'::text) = '삼성'::text)


---
## Step 4 · jsonb_path_ops 인덱스 크기 비교 (교안 5교시 모듈 5-3)

In [22]:
# jsonb_path_ops 방식의 GIN 인덱스를 doc_meta(5000행)에도 만들어, 기본 GIN과 크기를 비교합니다.
cur.execute("CREATE INDEX idx_meta_path ON doc_meta USING GIN (metadata jsonb_path_ops)")
conn.commit()

rows, cols = run("""
    SELECT
      pg_size_pretty(pg_relation_size('idx_meta_default')) AS default_gin_size,
      pg_size_pretty(pg_relation_size('idx_meta_path')) AS path_ops_size
""")
show(rows, cols)
# jsonb_path_ops는 @> 하나만 지원하는 대신(? 계열은 못 씀) 더 작고 약간 더 빠릅니다.

default_gin_size | path_ops_size
-----------------+--------------
112 kB           | 80 kB        


---
## 🔰 미션 · company 컬럼 승격 (교안 7교시 도전 2의 절반)

자주 조회하는 `company` 필드를 JSONB에서 꺼내 일반 컬럼으로 승격합니다.
`documents`(4건)에 적용합니다 — 7교시 도전 2에서 `doc_type`까지 이어서 확장합니다.

In [23]:
# 1) 컬럼 추가
# JSONB 안에 파묻혀 있던 company 값을, 자주 조회한다는 이유로 "일반 컬럼"으로 승격시킵니다.
# 일반 컬럼이 되면 표현식 인덱스 없이도 평범한 B-tree 인덱스를 걸 수 있어 더 단순하고 빠릅니다.
cur.execute("ALTER TABLE documents ADD COLUMN company TEXT")
conn.commit()

# 2) JSONB에서 기존 값 추출해 채우기 (백필, backfill)
# UPDATE ... SET company = metadata ->> 'company' : 이미 있던 모든 행에 대해, JSONB 안의 company
# 값을 꺼내 새로 만든 company 컬럼에 채워 넣습니다. "백필"이란 새 컬럼을 과거 데이터로 소급해서
# 채우는 작업을 뜻합니다.
cur.execute("UPDATE documents SET company = metadata ->> 'company'")
conn.commit()

rows, cols = run("SELECT id, title, company FROM documents ORDER BY id")
show(rows, cols)

id | title    | company
---+----------+--------
1  | 삼성 사업보고서 | 삼성     
2  | LG 감사보고서 | LG     
3  | 현대 반기보고서 | 현대     
4  | 복잡한 문서   |        


In [24]:
# 3) 무손실 검증 — 컬럼 값과 JSONB 값이 다른 행이 있는가? (0건이어야 정상)
# IS DISTINCT FROM : 일반적인 != 와 비슷하지만 NULL을 다르게 취급합니다. 보통의 != 는 NULL과 비교하면
# 결과가 NULL(알 수 없음)이 되어버려 놓치는 경우가 생기는데, IS DISTINCT FROM은 NULL도 "값"처럼 취급해
# 정확히 비교해줍니다 — 백필이 제대로 됐는지 빠짐없이 검증할 때 자주 쓰는 패턴입니다.
rows, cols = run("""
    SELECT count(*) AS mismatch
    FROM documents
    WHERE company IS DISTINCT FROM (metadata ->> 'company')
""")
show(rows, cols)

# 4) 컬럼에 인덱스
# 컬럼명만 적고 인덱스 이름을 생략하면, PostgreSQL이 "documents_company_idx" 같은 이름을 자동으로 붙여줍니다.
cur.execute("CREATE INDEX ON documents (company)")
conn.commit()
print("인덱스 생성 완료")

mismatch
--------
0       
인덱스 생성 완료


In [25]:
# 5) company='삼성' 조회 — @> 방식 vs 컬럼 방식 실행 계획 비교
print("=== @> 방식 ===")
explain('''SELECT * FROM documents WHERE metadata @> '{"company":"삼성"}' ''')

print()
print("=== 컬럼 방식 ===")
explain("SELECT * FROM documents WHERE company = '삼성'")
# 4건짜리 작은 테이블이라 Planner가 둘 다 Seq Scan을 고를 수 있습니다 — 정상입니다.
# (5교시 Step 3의 doc_meta 5,000행에서 이미 인덱스 효과 자체는 확인했습니다.)

=== @> 방식 ===
Seq Scan on documents  (cost=0.00..1.05 rows=1 width=100)
  Filter: (metadata @> '{"company": "삼성"}'::jsonb)

=== 컬럼 방식 ===
Seq Scan on documents  (cost=0.00..1.05 rows=1 width=100)
  Filter: (company = '삼성'::text)


---
## 보너스 · `jsonb_set`으로 부분 수정

JSONB 전체를 덮어쓰지 않고 특정 키 하나만 바꿉니다.

In [26]:
# 현대 반기보고서(id=3)의 pages를 220 → 235로 부분 수정
rows, cols = run("SELECT id, metadata ->> 'pages' AS pages_before FROM documents WHERE id = 3")
show(rows, cols)

# jsonb_set(대상 JSONB, 경로, 새 값) : JSONB 전체를 통째로 덮어쓰지 않고, 지정한 경로(키)의 값만
# 바꿔 끼워 넣은 "새 JSONB"를 만들어 돌려주는 함수입니다.
#   '{pages}' : 경로를 배열 형태로 지정 — 최상위의 pages 키를 가리킵니다(중첩된 경우 '{a,b,c}'처럼 여러 단계도 가능).
#   '235' : 새로 넣을 값. JSONB 함수이므로 이 값도 JSONB 형식의 문자열로 넘겨줍니다(숫자 235를 그대로 표현).
# 이 방식의 장점: 나머지 키(author, company, doc_type, tags 등)는 전혀 건드리지 않고 pages만 바뀝니다.
cur.execute("""
    UPDATE documents
    SET metadata = jsonb_set(metadata, '{pages}', '235')
    WHERE id = 3
""")
conn.commit()

rows, cols = run("SELECT id, metadata FROM documents WHERE id = 3")
show(rows, cols)

id | pages_before
---+-------------
3  | 220         
id | metadata                                                                                   
---+--------------------------------------------------------------------------------------------
3  | {'tags': ['공시', '반기'], 'pages': 235, 'author': '박지수', 'company': '현대', 'doc_type': '반기보고서'}


---
## 정리

- `documents`는 `course_db`에 영구 테이블로 남아 다음 주 pgvector 세션에서 `embedding` 컬럼이 추가됩니다.
- `doc_meta`는 인덱스 실험 전용이며 오늘 세션이 끝나도 별도로 지울 필요는 없지만 이후 세션에서 재사용하지 않습니다.
- 7교시 도전 1(정규화 vs JSONB 100건 비교)은 이 노트북에 없는 내용이라 교안의 힌트를 보고 직접 진행합니다.
- 7교시 도전 2는 위 🔰 미션에서 이어서 `doc_type`까지 확장하고, 선택 항목(JSONB에서 `company` 제거)까지 진행하면 완료입니다.